# RBSOG Benchmark Paper-Figure Notebook

该模板用于自动读取 `../results` 下最近一次实验产物，并生成论文风格图表：

1. `benchmark_summary.csv`：PPPM vs RBSOG 速度与方差对比图。
2. `batch_sweep_summary.csv/json`：批量大小 $P$ 的 Pareto 前沿与推荐点。
3. `stability_timeseries.csv` 与 `stability_summary.json`：结构稳定性时间序列与统计注释。

建议先执行：

- `rbsog-md benchmark ...`
- `rbsog-md sweep-batch ...`
- `rbsog-md run --solver rbsog ...`

然后按顺序运行本 Notebook 的全部代码单元。

In [ ]:
from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import numpy as np


ROOT = Path("..").resolve()
RESULTS_DIR = ROOT / "results"


def find_latest_file(pattern: str) -> Path:
    candidates = sorted(
        RESULTS_DIR.glob(pattern),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No file matches pattern: {pattern}")
    return candidates[0]


def read_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def to_float(value: str, default: float = float("nan")) -> float:
    try:
        return float(value)
    except Exception:
        return default


benchmark_summary_path = find_latest_file("**/benchmark_summary.csv")
sweep_summary_csv_path = find_latest_file("**/batch_sweep_summary.csv")
sweep_summary_json_path = find_latest_file("**/batch_sweep_summary.json")
stability_csv_path = find_latest_file("**/stability_timeseries.csv")
stability_summary_path = find_latest_file("**/stability_summary.json")

benchmark_rows = read_csv_rows(benchmark_summary_path)
sweep_rows = read_csv_rows(sweep_summary_csv_path)
with sweep_summary_json_path.open("r", encoding="utf-8") as handle:
    sweep_payload = json.load(handle)
stability_rows = read_csv_rows(stability_csv_path)
with stability_summary_path.open("r", encoding="utf-8") as handle:
    stability_summary = json.load(handle)

print("Benchmark summary:", benchmark_summary_path)
print("Sweep summary csv:", sweep_summary_csv_path)
print("Sweep summary json:", sweep_summary_json_path)
print("Stability csv:", stability_csv_path)
print("Stability summary:", stability_summary_path)
print("Rows:", len(benchmark_rows), len(sweep_rows), len(stability_rows))

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

solvers = [row["solver"].upper() for row in benchmark_rows]
step_time = np.array([to_float(row["mean_step_time"]) for row in benchmark_rows], dtype=float)
pressure_var = np.array([to_float(row["pressure_variance"]) for row in benchmark_rows], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2))
colors = ["#0f766e" if s == "RBSOG" else "#b45309" for s in solvers]

bars0 = axes[0].bar(solvers, step_time, color=colors)
axes[0].set_title("(A) Mean Step Time")
axes[0].set_ylabel("seconds")
for bar in bars0:
    h = bar.get_height()
    axes[0].annotate(f"{h:.4f}", (bar.get_x() + bar.get_width() / 2, h), ha="center", va="bottom", fontsize=9)

bars1 = axes[1].bar(solvers, pressure_var, color=colors)
axes[1].set_title("(B) Pressure Variance")
axes[1].set_ylabel("variance")
axes[1].set_yscale("log")
for bar in bars1:
    h = bar.get_height()
    axes[1].annotate(f"{h:.2e}", (bar.get_x() + bar.get_width() / 2, h), ha="center", va="bottom", fontsize=9)

solver_to_idx = {name: idx for idx, name in enumerate(solvers)}
if "PPPM" in solver_to_idx and "RBSOG" in solver_to_idx:
    pppm_t = step_time[solver_to_idx["PPPM"]]
    rbsog_t = step_time[solver_to_idx["RBSOG"]]
    pppm_v = pressure_var[solver_to_idx["PPPM"]]
    rbsog_v = pressure_var[solver_to_idx["RBSOG"]]
    speedup = pppm_t / max(rbsog_t, 1e-12)
    var_ratio = rbsog_v / max(pppm_v, 1e-12)
    fig.text(
        0.5,
        -0.02,
        f"Speedup (PPPM/RBSOG): {speedup:.3f}x    Variance ratio (RBSOG/PPPM): {var_ratio:.4f}",
        ha="center",
        fontsize=10,
    )

fig.suptitle("Solver Comparison", fontsize=14)
fig.tight_layout()
plt.show()

## Pareto Front And Recommended Batch Size

In [ ]:
sweep_rows_sorted = sorted(sweep_rows, key=lambda row: int(float(row["batch_size"])))

p_values = np.array([int(float(row["batch_size"])) for row in sweep_rows_sorted], dtype=int)
time_values = np.array([to_float(row["mean_step_time"]) for row in sweep_rows_sorted], dtype=float)
var_values = np.array([to_float(row["pressure_variance"]) for row in sweep_rows_sorted], dtype=float)
pareto_flags = np.array([to_float(row.get("is_pareto", "0")) > 0.5 for row in sweep_rows_sorted], dtype=bool)

recommended = sweep_payload.get("recommended_batch")
recommended_p = None if not isinstance(recommended, dict) else int(float(recommended["batch_size"]))

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.scatter(time_values, var_values, s=54, color="#0f766e", label="RBSOG points")

if np.any(pareto_flags):
    pareto_x = time_values[pareto_flags]
    pareto_y = var_values[pareto_flags]
    order = np.argsort(pareto_x)
    ax.plot(pareto_x[order], pareto_y[order], color="#155e75", linewidth=1.5, label="Pareto front")

for p, xv, yv in zip(p_values, time_values, var_values):
    ax.annotate(f"P={p}", (xv, yv), textcoords="offset points", xytext=(6, 5), fontsize=9)

baseline = sweep_payload.get("baseline_pppm", {})
if isinstance(baseline, dict):
    bx = float(baseline.get("mean_step_time", np.nan))
    by = float(baseline.get("pressure_variance", np.nan))
    if np.isfinite(bx) and np.isfinite(by):
        ax.scatter([bx], [by], s=150, marker="*", color="#b45309", label="PPPM baseline")

if recommended_p is not None:
    for p, xv, yv in zip(p_values, time_values, var_values):
        if int(p) != int(recommended_p):
            continue
        ax.scatter([xv], [yv], s=100, marker="D", color="#be123c", label=f"Recommended P={recommended_p}")
        ax.annotate(
            f"Recommended P={recommended_p}",
            (xv, yv),
            textcoords="offset points",
            xytext=(8, -14),
            fontsize=9,
            color="#be123c",
        )
        break

ax.set_xlabel("Mean Step Time (s)")
ax.set_ylabel("Pressure Variance")
ax.set_yscale("log")
ax.set_title("RBSOG Batch-Size Pareto Trade-off")
ax.grid(alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

if isinstance(recommended, dict):
    print(
        "Recommended batch:",
        f"P={int(float(recommended['batch_size']))},",
        f"utopia_score={float(recommended['utopia_score']):.4f},",
        f"speedup_vs_pppm={float(recommended['speedup_vs_pppm']):.3f},",
        f"variance_ratio_vs_pppm={float(recommended['variance_ratio_vs_pppm']):.4f}",
    )

## Structural Stability Time-Series (Publication Style)

In [ ]:
time = np.array([to_float(row["time"]) for row in stability_rows], dtype=float)
area = np.array([to_float(row["area_per_lipid"]) for row in stability_rows], dtype=float)
thickness = np.array([to_float(row["thickness_proxy"]) for row in stability_rows], dtype=float)
pressure = np.array([to_float(row.get("pressure", "nan")) for row in stability_rows], dtype=float)
temperature = np.array([to_float(row.get("temperature", "nan")) for row in stability_rows], dtype=float)


def moving_average(values: np.ndarray, window: int = 9) -> np.ndarray:
    if values.size == 0:
        return values
    window = max(1, int(window))
    if window == 1 or values.size < window:
        return values.copy()
    kernel = np.ones((window,), dtype=float) / float(window)
    return np.convolve(values, kernel, mode="same")


fig, axes = plt.subplots(2, 2, figsize=(11.2, 7.2), sharex=True)

axes[0, 0].plot(time, area, color="#0f766e", alpha=0.35, linewidth=1.2, label="raw")
axes[0, 0].plot(time, moving_average(area), color="#0f766e", linewidth=1.9, label="moving avg")
axes[0, 0].set_title("(A) Area Per Lipid")
axes[0, 0].set_ylabel("area")
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(time, thickness, color="#b45309", alpha=0.35, linewidth=1.2, label="raw")
axes[0, 1].plot(time, moving_average(thickness), color="#b45309", linewidth=1.9, label="moving avg")
axes[0, 1].set_title("(B) Membrane Thickness Proxy")
axes[0, 1].set_ylabel("thickness")
axes[0, 1].grid(alpha=0.3)
axes[0, 1].legend(fontsize=8)

axes[1, 0].plot(time, pressure, color="#155e75", linewidth=1.5)
axes[1, 0].set_title("(C) Instantaneous Pressure")
axes[1, 0].set_xlabel("time")
axes[1, 0].set_ylabel("pressure")
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(time, temperature, color="#7c2d12", linewidth=1.5)
axes[1, 1].set_title("(D) Instantaneous Temperature")
axes[1, 1].set_xlabel("time")
axes[1, 1].set_ylabel("temperature")
axes[1, 1].grid(alpha=0.3)

fig.suptitle("RBSOG Structural Stability Diagnostics", fontsize=14)
fig.tight_layout()
plt.show()

print("stability summary")
for key in [
    "area_per_lipid_mean",
    "area_per_lipid_std",
    "area_per_lipid_drift_per_time",
    "thickness_proxy_mean",
    "thickness_proxy_std",
    "thickness_proxy_drift_per_time",
]:
    if key in stability_summary:
        print(f"  {key}: {float(stability_summary[key]):.6e}")